# Data Collection: Building the Aviation Discourse Dataset
In this section, we set up the environment and import the necessary libraries for web scraping and text processing. Our goal is to collect historical Wikipedia data for US airports to analyze the shift in security narratives.

In [4]:
import pandas as pd
import requests
import re
import json
import time
import os
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# File configuration
OUTPUT_FILE = "AIRPORT_DATA_2001_2010.json"
AIRPORT_LIST_FILE = "airports_nodes.csv"
USER_AGENT = "DTU_Student_Project_Bot/1.4 (sXXXXXX@student.dtu.dk)"

## The Wikipedia Time Machine API
We define a function to fetch the specific version of a Wikipedia article as it appeared on the last day of a given year. This allows us to capture a 'snapshot' of the discourse at that exact moment in history.

In [5]:
def fetch_historical_text(session, title, iata, year):
    """
    Retrieves the historical text of a Wikipedia page closest to 
    the end of a specific year (Dec 31st).
    """
    api_url = "https://en.wikipedia.org/w/api.php"
    
    # We try two search terms: 1. The official name, 2. The IATA code as backup
    search_queries = [title, f"{iata} airport"]
    
    for query in search_queries:
        params = {
            "action": "query",
            "prop": "revisions",
            "titles": query,
            "redirects": 1,
            "rvlimit": 1,
            "format": "json",
            "rvprop": "content",
            "rvdir": "older",
            "rvstart": f"{year}-12-31T23:59:59Z"
        }
        
        try:
            response = session.get(api_url, params=params, timeout=10)
            data = response.json()
            pages = data.get("query", {}).get("pages", {})
            
            for page_id in pages:
                if page_id != "-1" and "revisions" in pages[page_id]:
                    return pages[page_id]["revisions"][0]["*"]
        except Exception as e:
            continue
        time.sleep(0.1) # Brief pause to respect Wikipedia's servers
    return ""

## Text Preprocessing
Raw Wikipedia text contains a lot of noise (markup, punctuation, etc.). We define a tokenization function that cleans the text, removes stop words, and prepares it for TF-IDF analysis.

In [6]:
def tokenize_text(text):
    """
    Cleans raw text by removing non-alphanumeric characters, 
    lowercasing, and filtering out common English stop words.
    """
    # Lowercase and remove symbols/markup
    clean_text = str(text).lower()
    clean_text = re.sub(r'[^a-z0-9\s]', '', clean_text)
    
    # Tokenize into individual words
    words = clean_text.split()
    
    # Define custom stopwords to filter out generic discourse
    stop_words = set(stopwords.words('english')).union({
        'also', 'may', 'one', 'would', 'use', 'used', 'external', 'links', 'ref'
    })
    
    return [w for w in words if w not in stop_words and len(w) > 2]

## Validation: Tracking Discourse Growth (2001-2025)
Before scraping 387 airports, we perform a pilot test on the 'Airport security' article to validate the 'Information Explosion' hypothesis. This confirms that the discourse matures significantly during our target period.

In [7]:
# --- CELL 4: PILOT TEST (VALIDATION) ---
# This cell tests our scraper on a single article ("Airport security")
# across a 25-year span to verify our data collection engine.

test_title = "Airport security"
pilot_results = {}

# We use a temporary session to keep the pilot test isolated
temp_session = requests.Session()
temp_session.headers.update({"User-Agent": USER_AGENT})

print(f"Starting Pilot Test for: {test_title}...", flush=True)

# Loop through years 2001 to 2025 to observe the historical growth
for test_year in range(2001, 2026):
    try:
        # Step 1: Fetch raw text from Wikipedia's history
        # (This uses the 'fetch_historical_text' function defined in Cell 2)
        raw_content = fetch_historical_text(temp_session, test_title, "N/A", test_year)
        
        # Step 2: Validate if content was found
        # We require at least 100 characters to consider it a valid article revision
        if raw_content and len(raw_content) > 100:
            
            # Step 3: Process the text into a complete token list
            # (This uses the 'tokenize_text' function defined in Cell 3)
            tokens = tokenize_text(raw_content)
            pilot_results[test_year] = tokens
            
            # Step 4: Display progress immediately
            print(f"Year {test_year}: Success! Found {len(tokens)} cleaned tokens.", flush=True)
        else:
            # If the page didn't exist or was a tiny 'stub'
            pilot_results[test_year] = []
            print(f"Year {test_year}: Article missing or too short.", flush=True)
            
    except Exception as e:
        # If a network error or other issue occurs, print the error and keep going
        print(f"Year {test_year}: Error encountered -> {e}", flush=True)
    
    # Wait 0.5 seconds to be polite to Wikipedia's API servers
    time.sleep(0.5) 

print("\n--- Pilot Test Complete! ---", flush=True)

Starting Pilot Test for: Airport security...
Year 2001: Article missing or too short.
Year 2002: Article missing or too short.
Year 2003: Success! Found 186 cleaned tokens.
Year 2004: Success! Found 407 cleaned tokens.
Year 2005: Success! Found 1088 cleaned tokens.
Year 2006: Success! Found 1464 cleaned tokens.
Year 2007: Success! Found 2223 cleaned tokens.
Year 2008: Success! Found 2986 cleaned tokens.
Year 2009: Success! Found 3215 cleaned tokens.
Year 2010: Success! Found 3455 cleaned tokens.
Year 2011: Success! Found 3440 cleaned tokens.
Year 2012: Success! Found 3675 cleaned tokens.
Year 2013: Success! Found 4002 cleaned tokens.
Year 2014: Success! Found 4099 cleaned tokens.
Year 2015: Success! Found 4124 cleaned tokens.
Year 2016: Success! Found 4129 cleaned tokens.
Year 2017: Success! Found 4206 cleaned tokens.
Year 2018: Success! Found 3944 cleaned tokens.
Year 2019: Success! Found 3975 cleaned tokens.
Year 2020: Success! Found 4372 cleaned tokens.
Year 2021: Success! Found 442

## Generating the Airport Node List
To know which airports to scrape, we first fetch a verified list of US airports from Wikipedia. This list will serve as the 'nodes' in our network analysis.

In [4]:
print("Fetching US airport list from Wikipedia...")
wiki_list_url = "https://en.wikipedia.org/wiki/List_of_airports_in_the_United_States"
browser_headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(wiki_list_url, headers=browser_headers)
tables = pd.read_html(response.text, match="IATA")
df_nodes = tables[0].dropna(subset=['IATA'])

# Cleaning the airport names for better API matching
def clean_airport_name(name):
    # Removes footnotes like [1] and text in parentheses (e.g. "Carl T. Jones Field")
    return re.sub(r'\(.*?\)|\[.*?\]', '', str(name)).strip().replace('–', '-')

df_nodes['Clean_Name'] = df_nodes['Airport'].apply(clean_airport_name)

# Save the node list for the Network Analysis part of the project
df_nodes[['IATA', 'City', 'Airport', 'Clean_Name']].to_csv(AIRPORT_LIST_FILE, index=False)
print(f"Success! Saved {len(df_nodes)} airports to {AIRPORT_LIST_FILE}.")

Fetching US airport list from Wikipedia...
Success! Saved 387 airports to airports_nodes.csv.


C:\Users\david\AppData\Local\Temp\ipykernel_34176\1156586568.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, match="IATA")
C:\Users\david\AppData\Local\Temp\ipykernel_34176\1156586568.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nodes['Clean_Name'] = df_nodes['Airport'].apply(clean_airport_name)


## The Main Loop: Scraping a Decade of History
We now iterate through every year (2001-2010) and every airport in our list. To prevent data loss during long runs, we save our progress every 50 airports.

In [5]:
# Initializing collection
master_data = []
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

airport_records = df_nodes.to_dict('records')
years_to_collect = range(2001, 2011)

print(f"Starting data collection for {len(airport_records) * len(years_to_collect)} possible entries...")
start_time = time.time()
processed_count = 0

for year in years_to_collect:
    print(f"\n--- Processing Year: {year} ---")
    for airport in airport_records:
        raw_content = fetch_historical_text(session, airport['Clean_Name'], airport['IATA'], year)
        tokens = tokenize_text(raw_content) if raw_content else []
        
        master_data.append({
            "Year": year,
            "IATA": airport['IATA'],
            "Airport": airport['Clean_Name'],
            "Tokens": tokens
        })
        
        processed_count += 1
        if processed_count % 50 == 0:
            # Intermediate save to protect progress
            with open(OUTPUT_FILE, "w", encoding='utf-8') as f:
                json.dump(master_data, f, indent=4)
            print(f"Progress Status: {processed_count} entries completed... Auto-saved.")

# Final Save
with open(OUTPUT_FILE, "w", encoding='utf-8') as f:
    json.dump(master_data, f, indent=4)

duration = round((time.time() - start_time) / 60, 1)
print(f"\nCOLLECTION COMPLETE! Total time: {duration} minutes.")

Starting data collection for 3870 possible entries...

--- Processing Year: 2001 ---
Progress Status: 50 entries completed... Auto-saved.
Progress Status: 100 entries completed... Auto-saved.
Progress Status: 150 entries completed... Auto-saved.
Progress Status: 200 entries completed... Auto-saved.
Progress Status: 250 entries completed... Auto-saved.
Progress Status: 300 entries completed... Auto-saved.
Progress Status: 350 entries completed... Auto-saved.

--- Processing Year: 2002 ---
Progress Status: 400 entries completed... Auto-saved.
Progress Status: 450 entries completed... Auto-saved.
Progress Status: 500 entries completed... Auto-saved.
Progress Status: 550 entries completed... Auto-saved.
Progress Status: 600 entries completed... Auto-saved.
Progress Status: 650 entries completed... Auto-saved.
Progress Status: 700 entries completed... Auto-saved.
Progress Status: 750 entries completed... Auto-saved.

--- Processing Year: 2003 ---
Progress Status: 800 entries completed... Au